# 오픈 모델 LoRA 파인튜닝 (KAN-58)

OpenAI가 새 파인튜닝 작업을 막아서(2026-11-15 `training_not_available`), 학습 JSONL을 그대로 써서 작은 오픈 모델을 Colab 무료 GPU(T4)에서 LoRA로 학습한다.

순서: ① 설치 → ② 파일 올리기 → ③ 학습 전 모델로 평가셋 답 만들기(base) → ④ LoRA 학습 → ⑤ 학습 후 답 만들기(lora) → ⑥ 결과 내려받기.
채점은 노트북 밖에서 한다: `node pipeline/scripts/score-answers.mjs prompts.jsonl answers.jsonl` (gpt-4.1 결과와 같은 채점).

**올릴 파일** (레포 밖 `reports/` 에 있음, 남의 자막이 들어 있어서 레포·공유 드라이브에 올리지 않기):
- `train-koen-v9.jsonl` 학습 84편
- `val-koen-v9.jsonl` 검증 10편
- `prompts.jsonl` 평가셋 37편 입력(정답 포함, `reports/fewshot-eval-2026-11-15/`)

런타임 → 런타임 유형 변경 → **T4 GPU** 를 먼저 고른다.

In [ ]:
# ① 설치 (2~3분)
!pip install -q unsloth
# unsloth가 지원하는 버전으로 고정한다. 최신 trl·datasets로 올리면 학습 칸에서 깨진다(11/16 확인).
!pip install -q "trl>=0.18.2,<=1.15.0" "datasets>=3.4.1,<4.0.0"

In [ ]:
# 설정. 메모리가 모자라면 MAX_LEN을 줄이거나 더 작은 모델(Qwen2.5-1.5B)로 바꾼다.
MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_LEN = 8192          # 입력+답 최대 토큰. 넘는 편은 학습에서 뺀다.
EPOCHS = 3
LR = 2e-4
LORA_R = 16
MAX_NEW = 2048          # 답 최대 토큰

In [ ]:
# ② 파일 올리기: train-koen-v9.jsonl, val-koen-v9.jsonl, prompts.jsonl
from google.colab import files
up = files.upload()
print(list(up))

In [ ]:
import json
read = lambda f: [json.loads(l) for l in open(f, encoding="utf-8") if l.strip()]
train = read("train-koen-v9.jsonl"); val = read("val-koen-v9.jsonl"); prompts = read("prompts.jsonl")
print(len(train), "train /", len(val), "val /", len(prompts), "eval prompts")

In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
lens = [len(tok.apply_chat_template(r["messages"], tokenize=True)) for r in train]
print("토큰 길이 최대", max(lens), "평균", sum(lens)//len(lens), "/ MAX_LEN 넘는 편", sum(l > MAX_LEN for l in lens))

In [ ]:
# ③·⑤ 평가셋 답 만들기. 답은 JSON 글자 그대로 저장하고, 채점 스크립트가 JSON을 읽는다.
import torch, time
def answer_all(label):
    FastLanguageModel.for_inference(model)
    out = []
    t0 = time.time()
    for i, p in enumerate(prompts):
        msgs = [m for m in p["messages"] if m["role"] != "assistant"]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
        if ids.shape[1] + MAX_NEW > MAX_LEN:
            out.append({"case": p["case"], "cond": label, "text": None, "note": "too long"}); continue
        with torch.no_grad():
            g = model.generate(input_ids=ids, max_new_tokens=MAX_NEW, do_sample=False)
        out.append({"case": p["case"], "cond": label, "text": tok.decode(g[0, ids.shape[1]:], skip_special_tokens=True)})
        print(f"{i+1}/{len(prompts)} {p['case']} {time.time()-t0:.0f}s")
    with open(f"answers-{label}.jsonl", "w", encoding="utf-8") as f:
        f.write("\n".join(json.dumps(o, ensure_ascii=False) for o in out) + "\n")
    return out

base_answers = answer_all("base")   # 학습 전

In [ ]:
# ④ LoRA 학습 (답 부분만 손실 계산)
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)
fmt = lambda rows: Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows
                                      if len(tok.apply_chat_template(r["messages"], tokenize=True)) <= MAX_LEN])
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=fmt(train), eval_dataset=fmt(val),
    args=SFTConfig(dataset_text_field="text", max_seq_length=MAX_LEN, per_device_train_batch_size=1,
        gradient_accumulation_steps=4, num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=5,
        lr_scheduler_type="linear", logging_steps=5, eval_strategy="epoch", save_strategy="no",
        fp16=True, optim="adamw_8bit", seed=0, report_to="none", output_dir="out"))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)

In [ ]:
# 학습 손실·검증 손실 기록 (발표용)
import pandas as pd
log = pd.DataFrame(trainer.state.log_history); log.to_csv("train-log.csv", index=False)
log[[c for c in ["step","epoch","loss","eval_loss"] if c in log]].dropna(how="all", subset=[c for c in ["loss","eval_loss"] if c in log])

In [ ]:
lora_answers = answer_all("lora")   # 학습 후
model.save_pretrained("lora-adapter"); tok.save_pretrained("lora-adapter")

In [ ]:
# ⑥ 결과 내려받기 → reports/fewshot-eval-2026-11-15/ 에 두고 채점
!zip -qr lora-adapter.zip lora-adapter
for f in ["answers-base.jsonl", "answers-lora.jsonl", "train-log.csv", "lora-adapter.zip"]:
    files.download(f)